# 05 Batch-level Logits Error Bound と Classification Margin の接続

## 今回の到達点

03 では、batch 全体の logits error に対して、

$$
\boxed{
\|\Delta L\|_F
\le
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
}
$$

を確認しました。

04 では、sample $n$ の logits error と classification margin に対して、

$$
\boxed{
\|\delta l_n\|_\infty
<
\frac{\gamma_n}{2}
}
$$

なら sample $n$ の predicted class が変わらないことを確認しました。

今回は、この2つを

$$
\boxed{
\|\delta l_n\|_\infty
\le
\|\delta l_n\|_2
\le
\|\Delta L\|_F
\le
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
}
$$

という1本の chain で接続します。

最終的に、

$$
\boxed{
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
<
\frac{\gamma_n}{2}
}
$$

なら、

$$
\boxed{
\operatorname*{argmax}_k
(\widetilde l_n)_k
=
\operatorname*{argmax}_k
(l_n)_k
}
$$

という sufficient condition を数値的に確認します。

## 今回扱わないこと

- batch 全体の minimum margin
- 全 sample を同時に保証する条件
- accuracy lower bound
- dataset 全体の accuracy
- Fashion-MNIST
- TT-rank sweep
- fine-tuning
- training / backward


## 1. 実行環境

初出変数は、以下の6項目で説明します。

1. 何を意味する変数か
2. 数式上の定義
3. shape
4. 各軸の意味
5. 何から計算されるか
6. 何のために使うか

| 変数 | 1. 意味 | 2. 数式上の定義 | 3. shape | 4. 各軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `dtype` | Tensor の数値型 | 数式対応なし | setting | 軸なし | `torch.float64` | 小さな誤差と不等式を安定して検証 |
| `device` | 計算デバイス | 数式対応なし | setting | 軸なし | CPU | 再現可能な toy 実験 |
| `rtol` | 相対許容誤差 | 数式対応なし | scalar | 軸なし | 手動設定 | `assert_close` 用 |
| `atol` | 絶対許容誤差 | 数式対応なし | scalar | 軸なし | 手動設定 | `assert_close` 用 |
| `eps` | 不等式用の微小 tolerance | 数式対応なし | scalar | 軸なし | 手動設定 | 浮動小数点誤差を考慮した `assert` |


In [1]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
from torch.testing import assert_close

torch.manual_seed(0)

dtype = torch.float64
device = torch.device("cpu")

rtol = 1e-10
atol = 1e-12
eps = 1e-12


## 2. 既習セットアップ：2層 MLP と logits error

ここは 01〜03 で扱った内容なので完成コードで用意します。

今回は手で追いやすい固定値を使います。

### shape

$$
X\in\mathbb R^{B\times d},
\qquad
W_1,\widetilde W_1,\Delta W_1\in\mathbb R^{h\times d},
$$

$$
W_2\in\mathbb R^{c\times h},
\qquad
L,\widetilde L,\Delta L\in\mathbb R^{B\times c}.
$$

| 変数 | 1. 意味 | 2. 数式上の定義 | 3. shape | 4. 各軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `x` | batch 入力 | $X$ | $(B,d)$ | batch × input | 固定 toy tensor | 第1 Linear への入力 |
| `w1` | dense 第1層 weight | $W_1$ | $(h,d)$ | hidden × input | 固定 toy tensor | dense forward |
| `delta_w1` | 第1層 weight error | $\Delta W_1$ | $(h,d)$ | hidden × input | 固定した小さい perturbation | 圧縮誤差の toy 代理 |
| `w1_tilde` | compressed 第1層 weight | $\widetilde W_1=W_1+\Delta W_1$ | $(h,d)$ | hidden × input | `w1 + delta_w1` | compressed forward |
| `w2` | 共通の第2層 weight | $W_2$ | $(c,h)$ | class × hidden | 固定 toy tensor | hidden activation を logits へ写す |
| `logits` | dense logits | $L$ | $(B,c)$ | batch × class | dense forward | margin の基準 |
| `logits_tilde` | compressed logits | $\widetilde L$ | $(B,c)$ | batch × class | compressed forward | 圧縮後 prediction |
| `delta_l` | batch 全体の logits error | $\Delta L=\widetilde L-L$ | $(B,c)$ | batch × class | `logits_tilde - logits` | sample error の母体 |
| `full_upper_bound` | 03で得た最終上界 | $\|X\|_F\|\Delta W_1\|_2\|W_2\|_2$ | scalar | 軸なし | 3つの norm の積 | $\|\Delta L\|_F$ を上から抑える |


In [2]:
B, d, h, c = 3, 2, 2, 3

x = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ],
    dtype=dtype,
    device=device,
)

w1 = torch.tensor(
    [
        [2.0, 0.0],
        [0.0, 2.0],
    ],
    dtype=dtype,
    device=device,
)

b1 = torch.tensor(
    [1.0, 1.0],
    dtype=dtype,
    device=device,
)

delta_w1 = torch.tensor(
    [
        [0.01, -0.005],
        [0.004, 0.008],
    ],
    dtype=dtype,
    device=device,
)

w1_tilde = w1 + delta_w1

w2 = torch.tensor(
    [
        [2.0, 0.0],
        [0.0, 1.5],
        [-1.0, -1.0],
    ],
    dtype=dtype,
    device=device,
)

b2 = torch.zeros(c, dtype=dtype, device=device)

# dense forward
z1 = x @ w1.T + b1
h_dense = torch.relu(z1)
logits = h_dense @ w2.T + b2

# compressed forward
z1_tilde = x @ w1_tilde.T + b1
h_tilde = torch.relu(z1_tilde)
logits_tilde = h_tilde @ w2.T + b2

# batch-level logits error
delta_l = logits_tilde - logits

# 03 の最終上界
x_fro = torch.linalg.matrix_norm(x, ord="fro")
delta_w1_spectral = torch.linalg.matrix_norm(delta_w1, ord=2)
w2_spectral = torch.linalg.matrix_norm(w2, ord=2)

full_upper_bound = (
    x_fro
    * delta_w1_spectral
    * w2_spectral
)

# 既習の定義チェック
delta_w1_from_weights = w1_tilde - w1

assert_close(
    delta_w1_from_weights,
    delta_w1,
    rtol=rtol,
    atol=atol,
)

assert x.shape == (B, d)
assert delta_w1.shape == (h, d)
assert logits.shape == (B, c)
assert logits_tilde.shape == (B, c)
assert delta_l.shape == (B, c)


## 3. $\delta l_n$ は $\Delta L$ の第 $n$ row

batch-level logits error は、

$$
\Delta L
=
\begin{bmatrix}
\delta l_1^\mathsf T\\
\delta l_2^\mathsf T\\
\vdots\\
\delta l_B^\mathsf T
\end{bmatrix}
\in
\mathbb R^{B\times c},
$$

と厳密に書けます。

ここでは各 sample の logits error を

$$
\delta l_n\in\mathbb R^c
$$

というベクトルとして表しているため、matrix $\Delta L$ の第 $n$ row は厳密には、

$$
\boxed{
(\Delta L)_{n,:}
=
\delta l_n^\mathsf T
}
$$

です。

ただし PyTorch の 1 次元 Tensor は shape `(c,)` であり、row vector / column vector の向きを区別しません。

そのため、この Notebook の実装では以降、

$$
\boxed{
\delta l_n
=
(\Delta L)_{n,:}
}
$$

と略記します。

これは、

```python
delta_l_n = delta_l[sample_index, :]
```

の shape が `(c,)` になることに対応しています。

今回は `sample_index = 2` の sample を対象にします。

| 変数 | 1. 意味 | 2. 数式上の定義 | 3. shape | 4. 各軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `sample_index` | 対象 sample の index | $n$ | Python int scalar | 軸なし | 固定値 2 | 1 sample を選ぶ |
| `delta_l_n` | sample $n$ の logits error | 厳密には $(\Delta L)_{n,:}=\delta l_n^\mathsf T$、実装では $\delta l_n=(\Delta L)_{n,:}$ と略記 | $(c,)$ | axis 0: class | `delta_l` の第 `sample_index` row | sample-wise norm を計算 |


In [ ]:
sample_index = 2

# TODO 1:
# Delta_L から sample_index 行を取り出してください。
delta_l_n = delta_l[sample_index, :]  # shape (c,)
print("delta_l_n:",delta_l_n.tolist())

delta_l_n: [0.009999999999999787, 0.017999999999999794, -0.01699999999999946]


### ヒント

`delta_l` の shape は `(B, c)` です。

欲しいのは第 `sample_index` row なので、結果の shape は `(c,)` になります。


### 検証

TODO 1 を埋めた後に実行してください。


In [7]:
assert delta_l_n.shape == (c,)

assert_close(
    delta_l_n,
    delta_l[sample_index, :],
    rtol=rtol,
    atol=atol,
)


## 4. Vector norm の関係

任意の vector

$$
v\in\mathbb R^c
$$

について、

$$
\|v\|_\infty
=
\max_k |v_k|
$$

です。

最大絶対値を持つ成分を $j$ とすると、

$$
|v_j|^2
\le
\sum_{k=1}^{c}v_k^2.
$$

両辺は非負なので平方根を取り、

$$
|v_j|
\le
\sqrt{
\sum_{k=1}^{c}v_k^2
}.
$$

左辺は $\|v\|_\infty$、右辺は $\|v\|_2$ なので、

$$
\boxed{
\|v\|_\infty
\le
\|v\|_2
}
$$

です。

したがって、

$$
\boxed{
\|\delta l_n\|_\infty
\le
\|\delta l_n\|_2
}
$$

となります。

| 変数 | 1. 意味 | 2. 数式上の定義 | 3. shape | 4. 各軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `delta_l_inf` | sample logits error の最大絶対成分 | $\|\delta l_n\|_\infty$ | scalar tensor | 軸なし | `delta_l_n` | 04の margin 条件と直接比較 |
| `delta_l_l2` | sample logits error の Euclidean norm | $\|\delta l_n\|_2$ | scalar tensor | 軸なし | `delta_l_n` | infinity norm と Frobenius norm の橋渡し |


In [ ]:
# TODO 2:
# delta_l_n の infinity norm と L2 norm を計算してください。
delta_l_inf = torch.max(torch.abs(delta_l_n))
delta_l_l2 = torch.linalg.vector_norm(delta_l_n)


### ヒント

- infinity norm:
  `torch.linalg.vector_norm(..., ord=float("inf"))`
- L2 norm:
  `torch.linalg.vector_norm(..., ord=2)`


### 検証

TODO 2 を埋めた後に実行してください。


In [9]:
assert delta_l_inf.shape == torch.Size([])
assert delta_l_l2.shape == torch.Size([])

assert delta_l_inf <= delta_l_l2 + eps


## 5. 1 row の $\ell_2$ norm と batch Frobenius norm

batch 全体では、

$$
\|\Delta L\|_F^2
=
\sum_{i=1}^{B}
\sum_{k=1}^{c}
(\Delta L)_{ik}^2.
$$

sample $n$ の row だけを見ると、

$$
\|\delta l_n\|_2^2
=
\sum_{k=1}^{c}
(\Delta L)_{nk}^2.
$$

Frobenius norm の二乗は、この row を含む **全 sample の非負な二乗和**なので、

$$
\|\delta l_n\|_2^2
\le
\|\Delta L\|_F^2.
$$

両辺は非負なので、

$$
\boxed{
\|\delta l_n\|_2
\le
\|\Delta L\|_F
}
$$

です。

| 変数 | 1. 意味 | 2. 数式上の定義 | 3. shape | 4. 各軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `delta_l_fro` | batch logits error の Frobenius norm | $\|\Delta L\|_F$ | scalar tensor | 軸なし | `delta_l` | sample-wise norm を上から抑える |


In [10]:
# TODO 3:
# batch 全体の Delta_L の Frobenius norm を計算してください。
delta_l_fro = torch.linalg.matrix_norm(delta_l,ord="fro")


### ヒント

`torch.linalg.matrix_norm(..., ord="fro")` を使います。


### 検証

TODO 3 を埋めた後に実行してください。


In [11]:
assert delta_l_fro.shape == torch.Size([])

assert delta_l_l2 <= delta_l_fro + eps

# 03 で得た batch-level upper bound も確認
assert delta_l_fro <= full_upper_bound + eps


## 6. Norm chain を1本につなぐ

ここまでで、

$$
\|\delta l_n\|_\infty
\le
\|\delta l_n\|_2
$$

および、

$$
\|\delta l_n\|_2
\le
\|\Delta L\|_F
$$

を得ました。

03 の結果、

$$
\|\Delta L\|_F
\le
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
$$

とつなぐと、

$$
\boxed{
\|\delta l_n\|_\infty
\le
\|\delta l_n\|_2
\le
\|\Delta L\|_F
\le
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
}
$$

です。

この chain の重要な点は、

> batch 全体に対する安全な上界を使って、1 sample の最大 component error まで安全側に上から抑えられる

ことです。


In [12]:
# TODO 4:
# 上の chain の4つの量を順に並べ、
# 全ての不等式が成り立つことを確認してください。
chain_holds = bool(
    (delta_l_inf <= delta_l_l2 + eps)
    & (delta_l_l2 <= delta_l_fro + eps)
    & (delta_l_fro <= full_upper_bound + eps)
)


### ヒント

確認するのは、

$$
\texttt{delta_l_inf}
\le
\texttt{delta_l_l2}
\le
\texttt{delta_l_fro}
\le
\texttt{full_upper_bound}
$$

です。


### 検証

TODO 4 を埋めた後に実行してください。


In [13]:
assert bool(chain_holds)

print(f"||delta l_n||_inf = {delta_l_inf.item():.8f}")
print(f"||delta l_n||_2   = {delta_l_l2.item():.8f}")
print(f"||Delta L||_F     = {delta_l_fro.item():.8f}")
print(f"full upper bound  = {full_upper_bound.item():.8f}")


||delta l_n||_inf = 0.01800000
||delta l_n||_2   = 0.02670206
||Delta L||_F     = 0.03997499
full upper bound  = 0.05221958


## 7. Sample $n$ の classification margin

04 では、

$$
y_n^\ast
=
\operatorname*{argmax}_k(l_n)_k
$$

とし、

$$
\gamma_n
=
(l_n)_{y_n^\ast}
-
\max_{k\neq y_n^\ast}(l_n)_k
$$

と定義しました。

今回は dense batch logits `logits` から対象 sample を取り出して margin を計算します。

| 変数 | 1. 意味 | 2. 数式上の定義 | 3. shape | 4. 各軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `l_n` | sample $n$ の dense logits | $l_n$ | $(c,)$ | axis 0: class | `logits[sample_index]` | dense prediction と margin |
| `l_tilde_n` | sample $n$ の compressed logits | $\widetilde l_n$ | $(c,)$ | axis 0: class | `logits_tilde[sample_index]` | compressed prediction |
| `y_star` | dense predicted class | $y_n^\ast$ | scalar integer tensor | 軸なし | `argmax(l_n)` | 維持したい class |
| `competitor_logit` | strongest competitor の logit | $\max_{k\ne y_n^\ast}(l_n)_k$ | scalar tensor | 軸なし | top class を除いた `l_n` | margin 計算 |
| `gamma_n` | classification margin | $(l_n)_{y_n^\ast}-\max_{k\ne y_n^\ast}(l_n)_k$ | scalar tensor | 軸なし | top logit と competitor logit | stability threshold |


In [19]:
l_n = logits[sample_index]
l_tilde_n = logits_tilde[sample_index]

# TODO 5:
# y_star, competitor_logit, gamma_n を計算してください。
from torch import argmax
y_star = argmax(l_n)
top_logit = l_n[y_star]
masked = l_n.clone()
masked[y_star] = float("-inf")   # 1位を除外
competitor_index = torch.argmax(masked)
competitor_logit = l_n[competitor_index]
gamma_n = top_logit - competitor_logit  # 1位 − 2位


### ヒント

04 と同じく、margin は「1位 − 2位」です。

`torch.topk` を使って top-2 logits を取得する方法でも構いません。


### 検証

TODO 5 を埋めた後に実行してください。


In [15]:
assert l_n.shape == (c,)
assert l_tilde_n.shape == (c,)
assert y_star.shape == torch.Size([])
assert competitor_logit.shape == torch.Size([])
assert gamma_n.shape == torch.Size([])

assert gamma_n > 0.0


## 8. TT weight error から prediction stability まで

04 の十分条件は、

$$
\|\delta l_n\|_\infty
<
\frac{\gamma_n}{2}.
$$

今回の chain から、

$$
\|\delta l_n\|_\infty
\le
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2.
$$

したがって、さらに強い条件、

$$
\boxed{
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
<
\frac{\gamma_n}{2}
}
$$

が成立すれば、自動的に、

$$
\|\delta l_n\|_\infty
<
\frac{\gamma_n}{2}
$$

です。

よって、

$$
\boxed{
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
<
\frac{\gamma_n}{2}
}
$$

なら、

$$
\boxed{
\operatorname*{argmax}_k
(\widetilde l_n)_k
=
\operatorname*{argmax}_k
(l_n)_k
}
$$

が保証されます。

これは、

$$
\text{TT weight error}
\rightarrow
\text{batch logits error bound}
\rightarrow
\text{sample logits error bound}
\rightarrow
\text{classification margin}
\rightarrow
\text{prediction stability}
$$

を初めて1本につないだ条件です。

| 変数 | 1. 意味 | 2. 数式上の定義 | 3. shape | 4. 各軸 | 5. 計算元 | 6. 用途 |
|---|---|---|---|---|---|---|
| `certificate_holds` | 理論上の stability certificate が成立したか | $\mathrm{upper\ bound}<\gamma_n/2$ | Python bool / scalar bool | 軸なし | `full_upper_bound`, `gamma_n` | prediction 不変を安全側に保証 |
| `y_tilde` | compressed predicted class | $\arg\max_k(\widetilde l_n)_k$ | scalar integer tensor | 軸なし | `l_tilde_n` | 実際の prediction と保証を比較 |


### TODO 6 の前に toy 設定の数値を確認する

この Notebook の固定 toy setting では、stability certificate が成立するように
$\Delta W_1$ を十分小さく設定しています。

TODO 6 を実装する前に、

$$
\|\Delta L\|_F,
\qquad
\|X\|_F\|\Delta W_1\|_2\|W_2\|_2,
\qquad
\frac{\gamma_n}{2}
$$

を実際に表示して、大小関係を確認します。

重要なのは、固定値を変更して certificate が成立しなくなっても、**理論が間違っているわけではない**ことです。
その場合は単に「この sufficient condition では prediction stability を保証できない」という意味です。


In [16]:
print(f"||Delta L||_F     = {delta_l_fro.item():.8f}")
print(f"full_upper_bound  = {full_upper_bound.item():.8f}")
print(f"gamma_n / 2       = {(gamma_n / 2).item():.8f}")


||Delta L||_F     = 0.03997499
full_upper_bound  = 0.05221958
gamma_n / 2       = 0.75000000


In [17]:
# TODO 6:
# full_upper_bound < gamma_n / 2 を確認し、
# compressed prediction y_tilde を計算してください。
certificate_holds = full_upper_bound < (gamma_n / 2)
y_tilde = torch.argmax(l_tilde_n)


### ヒント

ここでは、

- **理論保証**：`certificate_holds`
- **実際の予測結果**：`y_tilde == y_star`

を分けて考えます。

certificate が成立すれば、実際の argmax 不変も保証されます。


### 検証

TODO 6 を埋めた後に実行してください。


In [18]:
# この toy 設定では certificate が成立するように、
# delta_w1 を十分小さく選んでいます。
#
# 固定値を変更して certificate_holds == False になった場合でも、
# それは理論の失敗ではなく、
# 「この sufficient condition では保証できない」という意味です。
assert bool(certificate_holds)

assert y_tilde.shape == torch.Size([])
assert int(y_tilde) == int(y_star)

# certificate が成立しているので、
# 04 の sample-wise sufficient condition も満たされます。
assert delta_l_inf < gamma_n / 2

print(f"gamma_n / 2       = {(gamma_n / 2).item():.8f}")
print(f"full upper bound  = {full_upper_bound.item():.8f}")
print(f"certificate       = {bool(certificate_holds)}")
print(f"dense argmax      = {int(y_star)}")
print(f"compressed argmax = {int(y_tilde)}")


gamma_n / 2       = 0.75000000
full upper bound  = 0.05221958
certificate       = True
dense argmax      = 0
compressed argmax = 0


## 9. なぜこの条件は conservative なのか

今回の chain は、

$$
\|\delta l_n\|_\infty
\le
\|\delta l_n\|_2
\le
\|\Delta L\|_F
\le
\|X\|_F\|\Delta W_1\|_2\|W_2\|_2
$$

という複数の upper bound を重ねています。

そのため最終条件、

$$
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
<
\frac{\gamma_n}{2}
$$

は安全側ですが、かなり conservative になり得ます。

この条件を満たさなくても、

- sample $n$ の実際の error が batch Frobenius norm よりずっと小さい
- logits error の方向が argmax を壊す方向ではない
- 04で確認したように十分条件が不成立でも実際の argmax は維持される

場合があります。

したがって、

> certificate が False だから prediction が変わる

とは言えません。

言えるのは、

> certificate が True なら prediction が変わらない

です。


## 10. まとめ

今回の中心は、

$$
\boxed{
\|\delta l_n\|_\infty
\le
\|\delta l_n\|_2
\le
\|\Delta L\|_F
\le
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
}
$$

という chain です。

04 の margin condition、

$$
\|\delta l_n\|_\infty
<
\frac{\gamma_n}{2}
$$

と組み合わせることで、

$$
\boxed{
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
<
\frac{\gamma_n}{2}
}
$$

なら sample $n$ の predicted class が不変である、という十分条件を得ました。

今回の流れは、

$$
\boxed{
\Delta W_1
\rightarrow
\Delta L
\rightarrow
\delta l_n
\rightarrow
\gamma_n
\rightarrow
\text{argmax stability}
}
$$

です。

次は、

$$
\gamma_{\min}
=
\min_n \gamma_n
$$

を使って、batch 内の全 sample を同時に保証する条件へ進めます。

この Notebook ではまだそこへ進みません。
